In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import sys
sys.path.append("../src")
from pricing_engine import (load_model, recommend_price,
                             run_batch_pricing, PricingConstraints,
                             predict_demand, optimize_price)

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
model, feature_names = load_model(
    model_path   = "../models/xgb_demand_model.pkl",
    metrics_path = "../models/model_metrics.json"
)

In [ ]:
rec = recommend_price(
    model            = model,
    feature_names    = feature_names,
    product_id       = "P0001",
    base_price       = 1200.00,
    cost             = 800.00,
    competitor_price = 1150.00,
    inventory_ratio  = 0.08,
    demand_momentum  = 15.0,
    price_elasticity = -0.8
)

print(f"Current price    : ₹{rec['current_price']}")
print(f"Recommended price: ₹{rec['recommended_price']}")
print(f"Price change     : ₹{rec['price_change']} ({rec['price_change_pct']:+.1f}%)")
print(f"Price floor      : ₹{rec['price_floor']}")
print(f"Price ceiling    : ₹{rec['price_ceiling']}")
print(f"Predicted demand : {rec['predicted_demand']} units")
print(f"Expected revenue : ₹{rec['expected_revenue']}")
print(f"Revenue delta    : ₹{rec['revenue_delta']} ({rec['revenue_delta_pct']:+.1f}%)")
print(f"\nReasoning:")
for r in rec['reasoning']:
    print(f"  • {r['rule']}: {r['why']}")

In [ ]:
def plot_price_revenue_curve(model, feature_names, base_features, 
                              price_floor, price_ceiling, current_price,
                              recommended_price, title=""):
    prices   = np.linspace(price_floor, price_ceiling, 200)
    revenues = []
    demands  = []

    for p in prices:
        d = predict_demand(model, feature_names, base_features, p)
        revenues.append(p * d)
        demands.append(d)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Revenue curve
    axes[0].plot(prices, revenues, color="steelblue", linewidth=2)
    axes[0].axvline(current_price,      color="gray",   linestyle="--",
                    linewidth=1.5, label=f"Current ₹{current_price}")
    axes[0].axvline(recommended_price,  color="#1D9E75", linestyle="-",
                    linewidth=2,   label=f"Recommended ₹{recommended_price}")
    axes[0].fill_between(prices, revenues, alpha=0.08, color="steelblue")
    axes[0].set_xlabel("Price (₹)")
    axes[0].set_ylabel("Expected Revenue (₹)")
    axes[0].set_title(f"Price vs Revenue Curve\n{title}")
    axes[0].legend()

    # Demand curve
    axes[1].plot(prices, demands, color="coral", linewidth=2)
    axes[1].axvline(current_price,     color="gray",   linestyle="--",
                    linewidth=1.5, label=f"Current ₹{current_price}")
    axes[1].axvline(recommended_price, color="#1D9E75", linestyle="-",
                    linewidth=2,   label=f"Recommended ₹{recommended_price}")
    axes[1].set_xlabel("Price (₹)")
    axes[1].set_ylabel("Predicted Demand (units)")
    axes[1].set_title(f"Price vs Demand Curve\n{title}")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"../models/price_revenue_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: models/price_revenue_curve.png")


base_feats = {
    "base_price":        1200,
    "competitor_price":  1150,
    "inventory_ratio":   0.08,
    "demand_momentum":   15.0,
    "price_elasticity":  -0.8,
    "price_gap":         50,
    "price_gap_pct":     4.35,
    "is_cheaper":        0,
    "is_low_stock":      1,
    "is_critical_stock": 1,
}

plot_price_revenue_curve(
    model, feature_names, base_feats,
    price_floor        = rec["price_floor"],
    price_ceiling      = rec["price_ceiling"],
    current_price      = rec["current_price"],
    recommended_price  = rec["recommended_price"],
    title              = "P0001 — Critical Stock, Inelastic"
)

In [ ]:
sample_products = [
    {"product_id":"P0001","base_price":1200,"cost":800,
     "competitor_price":1150,"inventory_ratio":0.08,
     "demand_momentum":15.0,"price_elasticity":-0.8},
    {"product_id":"P0002","base_price":500,"cost":300,
     "competitor_price":480,"inventory_ratio":0.60,
     "demand_momentum":-5.0,"price_elasticity":-1.8},
    {"product_id":"P0003","base_price":250,"cost":150,
     "competitor_price":280,"inventory_ratio":0.20,
     "demand_momentum":2.0,"price_elasticity":-1.1},
    {"product_id":"P0004","base_price":3500,"cost":2200,
     "competitor_price":3600,"inventory_ratio":0.45,
     "demand_momentum":8.0,"price_elasticity":-0.5},
]

results_df = run_batch_pricing(model, feature_names, sample_products,
                               log_path="../data/processed/pricing_logs.csv")
results_df[["product_id","current_price","recommended_price",
            "price_change_pct","revenue_delta","revenue_delta_pct"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x     = range(len(results_df))
width = 0.35
ids   = results_df["product_id"].tolist()

bars1 = axes[0].bar([i - width/2 for i in x],
                    results_df["current_price"], width,
                    label="Current price", color="steelblue", alpha=0.8)
bars2 = axes[0].bar([i + width/2 for i in x],
                    results_df["recommended_price"], width,
                    label="Recommended price", color="#1D9E75", alpha=0.8)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(ids)
axes[0].set_ylabel("Price (₹)")
axes[0].set_title("Current vs Recommended Price")
axes[0].legend()

colors = ["#1D9E75" if v >= 0 else "#E24B4A"
          for v in results_df["revenue_delta"]]
axes[1].bar(ids, results_df["revenue_delta"], color=colors, alpha=0.85)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Revenue Impact (₹)")
axes[1].set_title("Expected Revenue Delta per Product")
for i, (val, pct) in enumerate(zip(results_df["revenue_delta"],
                                    results_df["revenue_delta_pct"])):
    axes[1].text(i, val + (5 if val >= 0 else -12),
                 f"{pct:+.1f}%", ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for i, row in results_df.iterrows():
    y = i
    ax.plot([row["price_floor"], row["price_ceiling"]], [y, y],
            color="lightgray", linewidth=8, solid_capstyle="round", zorder=1)
    ax.scatter(row["current_price"],     y, color="steelblue",
               s=120, zorder=3, label="Current" if i==0 else "")
    ax.scatter(row["recommended_price"], y, color="#1D9E75",
               s=120, marker="D", zorder=3, label="Recommended" if i==0 else "")
    ax.scatter(row["competitor_price"],  y, color="coral",
               s=100, marker="^", zorder=3, label="Competitor" if i==0 else "")

ax.set_yticks(range(len(results_df)))
ax.set_yticklabels(results_df["product_id"])
ax.set_xlabel("Price (₹)")
ax.set_title("Pricing Boundaries: Floor → Ceiling with Recommendations")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()